# Step 2 - Training a Simple Model
## Semantic Segmentation with Deep Learning - Potsdam Dataset

**Input:** RGB + IR (4 bands)  
**Training:** 20 epochs, Categorical Cross-Entropy, Best validation model saved  
**Data augmentation:** Rotation + Flipping

## 2.1 Install and Import Libraries

In [ ]:
!pip install rasterio tensorflow matplotlib scikit-learn -q

In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import visualkeras

# =====================================================================
# Hyperparameters
# =====================================================================
SEED           = 42
BATCH_SIZE     = 16
EPOCHS         = 20
LEARNING_RATE  = 1e-3
NUM_CLASSES    = 6
INPUT_CHANNELS = 4          # RGB + IR
PATCH_SIZE     = 64         # tile height & width (assumption)
MODEL_SAVE_PATH = 'best_simple_model.h5'

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('Libraries imported!')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## 2.2 Load Fold Splits

In [ ]:
# =====================================================================
# CONFIGURATION - Adjust paths as needed
# =====================================================================
# Original local path:
# DATA_DIR = r'c:\Users\mina_\OneDrive\Documents\DESING_OF_AI_SYSTEMS\Semantic Segmentation with Deep Learning\PROJECT\Potsdam-GeoTif'

import os
POSSIBLE_DATA_PATHS = [
    os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.path.dirname(os.getcwd()), 'Potsdam-GeoTif'),
    os.path.join(os.path.dirname(os.getcwd()), 'data'),
    os.getcwd()
]
DATA_DIR = next((p for p in POSSIBLE_DATA_PATHS if os.path.exists(p)), 'data')
print(f"Data directory: {DATA_DIR}")
# =====================================================================

# Load the fold splits saved in Step 1
# with open('fold_splits.json', 'r') as fp:
with open(os.path.join(DATA_DIR, 'fold_splits.json'), 'r') as fp:
    fold_splits = json.load(fp)

train_files = fold_splits['train']
val_files   = fold_splits['val']
test_files  = fold_splits['test']

print(f'Training files  : {len(train_files)}')
print(f'Validation files: {len(val_files)}')
print(f'Test files      : {len(test_files)}')

## 2.3 Data Generator with Augmentation

In [ ]:
CLASS_COLORS = [
    [255, 255, 255],  # Impervious surface
    [0,   0,   255],  # Building
    [0,   255,   0],  # Tree
    [0,   255, 255],  # Low vegetation
    [255, 255,   0],  # Car
    [255,   0,   0],  # Clutter/Background
]

def read_geotiff(file_path):
    with rasterio.open(file_path) as src:
        return src.read()  # (bands, H, W)

def normalize_band(band):
    b_min, b_max = band.min(), band.max()
    if b_max == b_min:
        return np.zeros_like(band, dtype=np.float32)
    return (band - b_min) / (b_max - b_min)

def load_sample(file_path, use_all_bands=False):
    """Load features and one-hot labels from a GeoTIFF."""
    data = read_geotiff(file_path)
    
    n_bands = 5 if use_all_bands else 4
    features = data[:n_bands].transpose(1, 2, 0).astype(np.float32)  # (H, W, C)
    
    for c in range(features.shape[-1]):
        features[..., c] = normalize_band(features[..., c])
    
    label_band = data[5].astype(np.int32)  # (H, W)
    label_onehot = tf.keras.utils.to_categorical(label_band, num_classes=NUM_CLASSES)
    
    return features, label_onehot


def augment(image, label):
    """Apply data augmentation: rotation and flipping."""
    # Random horizontal flip
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        label = tf.image.flip_left_right(label)
    
    # Random vertical flip
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        label = tf.image.flip_up_down(label)
    
    # Random 90-degree rotation (0, 90, 180, 270)
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    label = tf.image.rot90(label, k)
    
    return image, label


def make_dataset(file_paths, use_all_bands=False, augment_data=False, batch_size=16, shuffle=True):
    """Create a tf.data.Dataset from a list of file paths."""
    
    def _load(fp):
        fp_str = fp.numpy().decode('utf-8')
        features, labels = load_sample(fp_str, use_all_bands=use_all_bands)
        return features, labels
    
    def _tf_load(fp):
        features, labels = tf.py_function(
            _load, [fp],
            [tf.float32, tf.float32]
        )
        return features, labels
    
    ds = tf.data.Dataset.from_tensor_slices(file_paths)
    
    if shuffle:
        ds = ds.shuffle(buffer_size=len(file_paths), seed=SEED)
    
    ds = ds.map(_tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


print('Dataset functions defined!')

In [ ]:
# Create training, validation, and test datasets
train_ds = make_dataset(train_files, use_all_bands=False, augment_data=True,  batch_size=BATCH_SIZE, shuffle=True)
val_ds   = make_dataset(val_files,   use_all_bands=False, augment_data=False, batch_size=BATCH_SIZE, shuffle=False)
test_ds  = make_dataset(test_files,  use_all_bands=False, augment_data=False, batch_size=BATCH_SIZE, shuffle=False)

print('Datasets created!')

## 2.4 Simple CNN Model Architecture

A simple fully-convolutional network for pixel-wise classification.

In [ ]:
def build_simple_model(input_channels=4, num_classes=6):
    """
    Simple CNN for semantic segmentation.
    Architecture: Conv → BN → ReLU (repeated) → 1x1 Conv → Softmax
    """
    inputs = keras.Input(shape=(None, None, input_channels), name='input')
    
    # Encoder block 1
    x = layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1_1')(inputs)
    x = layers.BatchNormalization(name='bn1_1')(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1_2')(x)
    x = layers.BatchNormalization(name='bn1_2')(x)
    
    # Encoder block 2
    x = layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2_1')(x)
    x = layers.BatchNormalization(name='bn2_1')(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2_2')(x)
    x = layers.BatchNormalization(name='bn2_2')(x)
    
    # Encoder block 3
    x = layers.Conv2D(128, 3, padding='same', activation='relu', name='conv3_1')(x)
    x = layers.BatchNormalization(name='bn3_1')(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu', name='conv3_2')(x)
    x = layers.BatchNormalization(name='bn3_2')(x)
    
    # Output layer: 1x1 convolution for pixel-wise classification
    outputs = layers.Conv2D(num_classes, 1, padding='same', activation='softmax', name='output')(x)
    
    model = keras.Model(inputs, outputs, name='SimpleCNN_SegModel')
    return model


# Build and summarize the model
model = build_simple_model(input_channels=INPUT_CHANNELS, num_classes=NUM_CLASSES)
model.summary()

## 2.5 Visualize Model Architecture

In [ ]:
# Install visualkeras for visualization
try:
    import visualkeras
    img = visualkeras.layered_view(model, legend=True, to_file='simple_model_architecture.png')
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Simple CNN Model Architecture')
    plt.show()
    print('Architecture saved to simple_model_architecture.png')
except Exception:
    # Fallback: plot with Keras
    keras.utils.plot_model(model, to_file='simple_model_architecture.png',
                           show_shapes=True, show_layer_names=True, dpi=80)
    from IPython.display import Image, display
    display(Image('simple_model_architecture.png'))
    print('Architecture saved to simple_model_architecture.png')

## 2.6 Model Compilation and Training

In [ ]:
# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
callbacks = [
    ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

print(f'Model compiled. Training for {EPOCHS} epochs...')
print(f'Best model will be saved to: {MODEL_SAVE_PATH}')

In [ ]:
# Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 2.7 Plot Training History

In [ ]:
def plot_history(history, title_prefix=''):
    """Plot training and validation loss and accuracy curves."""
    epochs_range = range(1, len(history.history['loss']) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    ax1.plot(epochs_range, history.history['loss'],     label='Training Loss',   color='blue')
    ax1.plot(epochs_range, history.history['val_loss'], label='Validation Loss', color='orange')
    ax1.set_title(f'{title_prefix} Loss Curve')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy
    ax2.plot(epochs_range, history.history['accuracy'],     label='Training Accuracy',   color='green')
    ax2.plot(epochs_range, history.history['val_accuracy'], label='Validation Accuracy', color='red')
    ax2.set_title(f'{title_prefix} Accuracy Curve')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{title_prefix.replace(" ", "_")}_training_curves.png', dpi=150)
    plt.show()
    print(f'Training curves saved.')

plot_history(history, title_prefix='Simple CNN')

## 2.8 Evaluate on Test Set

In [ ]:
# Load best saved model
best_model = keras.models.load_model(MODEL_SAVE_PATH)

# Evaluate on test set
print('Evaluating on test set...')
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=1)

print(f'\n===== Test Set Results =====')
print(f'Test Loss    : {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

## 2.9 Summary

✅ Simple CNN model built with 3 encoder blocks  
✅ Trained for 20 epochs with RGB + IR (4 bands)  
✅ Data augmentation applied (rotation, flipping)  
✅ Best model saved based on validation accuracy  
✅ Training curves plotted  
✅ Model evaluated on test set  

**Next Step:** Proceed to `Step3_Encoder_Decoder_Model.ipynb`